# RASCAR V15 — воспроизведение сабмита

Распакуйте `rascar_v15_bundle.zip` и откройте ноутбук внутри `rascar_v15`. Этот запуск повторяет постобработку из сохранённых предсказаний моделей и проверяет SHA256 финального CSV. GPU и интернет не требуются.

Новые модели не обучаются повторно. Код обучения и извлечения признаков, три веса YOLO и fitted DINO-heads находятся рядом. Описание ограничений — в `SUMMARY.txt`.

Для окружения: `pip install -r requirements_replay.txt`.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib
A = Path.cwd()
if not (A / 'common.py').is_file():
    A = A / 'rascar_v15'
assert (A / 'common.py').is_file(), 'Откройте ноутбук в папке rascar_v15'
os.environ['RASCAR_V15_ROOT'] = str(A.resolve())
print(A.resolve())

## Постобработка

Дни, геометрия, семантика, гипотезы высоты, локальные кисти, группы дублей и pose. Все входные предсказания уже сохранены. Выбор окончательной конфигурации зафиксирован в `finalize.py`.

In [ ]:
scripts = ['person_ablation.py', 'evaluate_components.py', 'person_hypotheses.py', 'duplicate_groups.py', 'combine.py', 'orientation_guard.py', 'evaluate_pose.py', 'evaluate_hand_specialist.py', 'finalize.py', 'verify_final.py']
for script in scripts:
    result = subprocess.run([sys.executable, str(A/script)], cwd=A, text=True, capture_output=True)
    if result.returncode:
        print(result.stdout, result.stderr)
        raise RuntimeError(script)
    print(script, 'OK')
assert hashlib.sha256((A/'submission_v15.csv').read_bytes()).hexdigest() == 'f300490cc6555e8bbde670e4bf5ad18873f0cc8cd087043cc8a697bcd07c2e22'

In [ ]:
import pandas as pd
names=['baseline_v8','baseline_v14','final_hands0.2_pose0.2']
display(pd.DataFrame([json.loads((A/'results'/f'{n}.json').read_text()) for n in names])[['experiment','pooled_ap','day_25','day_26','public_lb']])
print((A/'final_verification.json').read_text())
from IPython.display import FileLink
display(FileLink(str(A/'submission_v15.csv')))
display(FileLink(str(A/'REPORT.html')))

## Обучение моделей — отдельный этап

`train_hand_specialist.py` принимает `2026-10-25`, `2026-10-26` или `all` и номер GPU. Нужны исходные изображения, `image_paths.csv` с колонками frame_id/path и оригинальные pretrained `weights/yolo26s.pt`. Веса уже обученных специалистов сохранены отдельно в `weights/hand25_last.pt`, `hand26_last.pt`, `handall_last.pt`.

`extract_semantics.py` использует gated DINOv3 и Kaggle secret HF_TOKEN; `extract_pose.py` — официальный YOLO26m-pose. `prepare_remote.py` — фактически использованная подготовка Kaggle. Часть test-изображений пришлось восстановить из ранее сохранённого аудита после HTTP 404; это отражено в журнале.

Повторное обучение может немного отличаться из-за окружения и GPU; точное воспроизведение выданного CSV обеспечивает сохранённый кэш. Значения обучения/валидации из Ultralytics train/results.csv относятся к обучающему набору, поскольку val=train и val=False; итоговая независимая проверка — только cross-day AP из results/oof.